# Map each genus to Uniprot Proteome ID

## Configuration

In [1]:
import pandas as pd
import os
import json
import requests
import pickle
from difflib import SequenceMatcher

In [28]:
filt_d = "./data/FILTERED_S.xlsx"
genus_f = "./data/Students_Samples_abundance_FILTERED.csv"

human_brain = "./data/gut_c3/Human_proteins_brain_f.csv"
human_gut = "./data/gut_c3/Human_proteins_gut_f.csv"

## Find Proteome IDs mapped from clinical data
* **by hand**

* **from hgma**


In [3]:
# Read filt_d: only need the first two sheets (hand_mapped and hgma)
filt_df_1 = pd.read_excel(filt_d, index_col=None, header=None, sheet_name="hand_mapped")
# Add column names
filt_df_1.columns = ["Genus", "Proteome ID"]

filt_df_2 = pd.read_excel(filt_d, index_col=None, header=0, sheet_name="hgma")
# Only keep name and proteome ID columns
filt_df_2 = filt_df_2[["name", "Proteome ID(with higher BUSCO)"]]
filt_df_2.columns = ["Species", "Proteome ID"]


In [4]:
filt_df_1

,Genus,Proteome ID
0,g__Alloscardovia,UP000016519
1,g__Arcanobacterium,UP000293036
2,g__Brevibacterium,UP000031488
3,g__CAG-352,UP000018340
4,g__Christensenellaceae R-7 group,UP000531233
...,...,...
58,g__unknown_ Saccharimonadales,UP000013893
59,g__unknown_ Synergistaceae,UP000259231
60,g__unknown_ Veillonellaceae,UP000070542
61,g__unknown_ Victivallaceae,UP000245959


In [5]:
filt_df_2

,Species,Proteome ID
0,Acidaminococcus fermentans,UP000182379
1,Acidaminococcus intestini,UP000754226
2,Acidaminococcus sp. CAG:542,UP000018195
3,Actinomyces oris,UP000185603
4,Adlercreutzia equolifaciens,UP000472380
...,...,...
410,Peptoniphilus harei,UP000250070
411,Peptoniphilus lacrimalis,UP000255517
412,unclassified Prevotellaceae,UP000182805
413,unclassified Rhodospirillales,UP000438537


## Create taxonomy list for study

In [6]:
# Read genus_f
genus_df = pd.read_csv(genus_f, index_col=0)

# Taxonomy list from column name
tax_list = list(genus_df["Taxonomy"])
tax_list

['g__Acidaminococcus',
 'g__Actinomyces',
 'g__Adlercreutzia',
 'g__Aeromonas',
 'g__Akkermansia',
 'g__Alistipes',
 'g__Allisonella',
 'g__Alloprevotella',
 'g__Alloscardovia',
 'g__AM778010_g',
 'g__Anaeroplasma',
 'g__Anaerostipes',
 'g__Anaerotruncus',
 'g__Arcanobacterium',
 'g__Asteroleplasma',
 'g__Atopobium',
 'g__Bacteroides',
 'g__Bacteroides pectinophilus group',
 'g__Barnesiella',
 'g__Bifidobacterium',
 'g__Bilophila',
 'g__Blautia',
 'g__Brevibacterium',
 'g__Butyricicoccus',
 'g__Butyricimonas',
 'g__Butyrivibrio',
 'g__CAG-352',
 'g__Campylobacter',
 'g__Catenibacterium',
 'g__Christensenellaceae R-7 group',
 'g__Cloacibacillus',
 'g__Clostridium innocuum group',
 'g__Clostridium sensu stricto 1',
 'g__Clostridium sensu stricto 13',
 'g__Clostridium sensu stricto 2',
 'g__Colidextribacter',
 'g__Collinsella',
 'g__Comamonas',
 'g__Coprobacillus',
 'g__Coprobacter',
 'g__Coprococcus',
 'g__Corynebacterium',
 'g__Cutibacterium',
 'g__Dermabacter',
 'g__Desulfovibrio',
 'g

## Map Species names from hgma to Taxonomy

In [7]:
# For every entry of filt_df_2, compare simillarity of species name with genus name from tax_list

# Remove g__ (3 characters) from tax_list
tax_list_n = []
for i in range(len(tax_list)):
    tax_list_n.append(tax_list[i][3:])
match_genus_l = []
for i in range(len(filt_df_2)):
    score_list = []
    for j in range(len(tax_list_n)):
            # Calculate sequence similarity score
            score = SequenceMatcher(None, filt_df_2["Species"][i], tax_list_n[j]).ratio()
            # Append to score_list with scores for every entry
            score_list.append(score)
    # Find index of highest score
    index = score_list.index(max(score_list))
    # Find genus of highest score
    match_genus = tax_list[index]
    # Append to match_genus_l
    match_genus_l.append(match_genus)
    print(filt_df_2["Species"][i], "/", match_genus)
# Add match_genus_l to filt_df_2
filt_df_2["Genus"] = match_genus_l
# Drop species column
filt_df_2 = filt_df_2.drop(columns=["Species"])

Acidaminococcus fermentans / g__Acidaminococcus
Acidaminococcus intestini / g__Acidaminococcus
Acidaminococcus sp. CAG:542 / g__Acidaminococcus
Actinomyces oris / g__Actinomyces
Adlercreutzia equolifaciens / g__Adlercreutzia
Aeromonas caviae / g__Aeromonas
Aeromonas veronii / g__Aeromonas
Akkermansia muciniphila / g__Akkermansia
Akkermansia sp. KLE1605 / g__Akkermansia
unclassified Akkermansia / g__Akkermansia
unclassified Akkermansiaceae / g__Akkermansia
Alistipes finegoldii / g__Alistipes
Alistipes indistinctus / g__Alistipes
Alistipes obesi / g__Alistipes
Alistipes onderdonkii / g__Alistipes
Alistipes putredinis / g__Alistipes
Alistipes shahii / g__Alistipes
Alistipes sp. An31A  / g__Alistipes
Alistipes sp. CAG:268 / g__Alistipes
Alistipes sp. CHKCI003 / g__Alistipes
Alistipes timonensis / g__Alistipes
unclassified Alistipes / g__Alistipes
Allisonella histaminiformans / g__Allisonella
Alloprevotella tannerae / g__Alloprevotella
unclassified_Anaeroplasmataceae / g__Anaeroplasma
Anaer

## Merge filt_df_1 and filt_df_2 : Final match of Genus to Proteome ID

In [8]:
# Concatenate filt_df_1 and filt_df_2
filt_df = pd.concat([filt_df_1, filt_df_2], axis=0)
# Drop duplicates
filt_df = filt_df.drop_duplicates()
filt_df

,Genus,Proteome ID
0,g__Alloscardovia,UP000016519
1,g__Arcanobacterium,UP000293036
2,g__Brevibacterium,UP000031488
3,g__CAG-352,UP000018340
4,g__Christensenellaceae R-7 group,UP000531233
...,...,...
408,g__unknown_ Parvimonas,UP000031386
409,g__unknown_ Parvimonas,UP000070197
410,g__unknown_ Peptoniphilus,UP000250070
411,g__unknown_ Peptoniphilus,UP000255517


In [9]:
os.makedirs("./data/proteomes", exist_ok=True)
tot = 0
for k, v in filt_df.iterrows():
    print(k, v["Proteome ID"])
    protFile = "./data/proteomes/" + "{}.pkl".format(v["Proteome ID"])
                    
    if not os.path.isfile(protFile):
        reviewed = list()

        req = "https://www.ebi.ac.uk/proteins/api/proteomes/proteins/{}?reviewed=True".format(v["Proteome ID"])
        r = requests.get(req)
        res = r.json()

        if "component" in res and len(res["component"]) > 0:
            if "protein" in res["component"][0]:
                for p in res["component"][0]["protein"]:
                    reviewed.append(p["accession"])
        
        unreviewed = list()

        req = "https://www.ebi.ac.uk/proteins/api/proteomes/proteins/{}?reviewed=False".format(v["Proteome ID"])
        r = requests.get(req)
        res = r.json()
        if "component" in res and len(res["component"]) > 0:
            if "protein" in res["component"][0]:
                for p in res["component"][0]["protein"]:
                    unreviewed.append(p["accession"])
        
        protList = reviewed + unreviewed
        protList = list(set(protList))

        with open(protFile, "wb") as fp:   
            pickle.dump(protList, fp)

0 UP000016519
1 UP000293036
2 UP000031488
3 UP000018340
4 UP000531233
5 UP000013051
6 UP000461734
7 UP000033457
8 UP000005332
9 UP000014577
10 UP000199355
11 UP000003277
12 UP000528555
13 UP000001006
14 UP000018069
15 UP000018868
16 UP000483027
17 UP000253976
18 UP000007468
19 UP000295711
20 UP000033124
21 UP000176615
22 UP000074544
23 UP000068137
24 UP000053091
25 UP000003045
26 UP000591941
27 UP000298358
28 UP000031461
29 UP000027946
30 UP000198995
31 UP000255233
32 UP000033103
33 UP000242432
34 UP000018843
35 UP000561134
36 UP000003821
37 UP000215624
38 UP000531233
39 UP000824078
40 UP000190657
41 UP000245793
42 UP000556020
43 UP000831151
44 UP000001319
45 UP000581150
46 UP000004191
47 UP000199158
48 UP000514720
49 UP000191179
50 UP000233535
51 UP000576402
52 UP000678348
53 UP000681387
54 UP000006605
55 UP000182805
56 UP000525652
57 UP000438537
58 UP000013893
59 UP000259231
60 UP000070542
61 UP000245959
62 UP000016618
0 UP000182379
1 UP000754226
2 UP000018195
3 UP000185603
4 UP00047

In [12]:
filt_df["Proteome ID"].nunique()

455

In [11]:
filt_df["Genus"].nunique()

165

In [13]:
# Construct the bac_info_dir. Key1: genus, key2: proteome ID and value: list of proteins

bac_info_dir = {}
for k, v in filt_df.iterrows():
    if v["Genus"] in bac_info_dir:
        bac_info_dir[v["Genus"]][v["Proteome ID"]] = list()
    else:
        bac_info_dir[v["Genus"]] = {}
        bac_info_dir[v["Genus"]][v["Proteome ID"]] = list()
    protFile = "./data/proteomes/" + "{}.pkl".format(v["Proteome ID"])
    with open(protFile, "rb") as fp:
        protList = pickle.load(fp)
    bac_info_dir[v["Genus"]][v["Proteome ID"]] = protList


# Save to json
with open("./data/bac_info.json", "w") as fp:
    json.dump(bac_info_dir, fp, indent=4)

In [14]:
proteins = set()
for genus, proteome_dict in bac_info_dir.items():
    print(genus, len(proteome_dict))
    for proteome, prot_list in proteome_dict.items():
        print(proteome, len(prot_list))
        proteins.update(prot_list)
print(len(proteins))

g__Alloscardovia 1
UP000016519 1649
g__Arcanobacterium 1
UP000293036 1664
g__Brevibacterium 1
UP000031488 3434
g__CAG-352 1
UP000018340 0
g__Christensenellaceae R-7 group 1
UP000531233 1884
g__Clostridium innocuum group 1
UP000013051 4653
g__Colidextribacter 1
UP000461734 3624
g__Corynebacterium 2
UP000033457 2047
UP000029601 1151
g__Cutibacterium 1
UP000005332 2376
g__Dermabacter 1
UP000014577 2120
g__Desulfovibrio 8
UP000199355 2277
UP000297065 2637
UP000319704 0
UP000069241 3007
UP000186323 2530
UP000195838 2772
UP000246125 3095
UP000014960 3444
g__Dialister 1
UP000003277 2140
g__Dorea 1
UP000528555 2205
g_Shigella 1
UP000001006 3859
g__Eubacterium hallii group 2
UP000018069 0
UP000014176 3773
g__Eubacterium nodatum group 3
UP000018868 1690
UP000002521 2046
UP000055238 0
g__Facklamia 1
UP000483027 1842
g__Family XIII AD3011 group 1
UP000253976 1520
g__Filifactor 1
UP000007468 1615
g__Frisingicoccus 1
UP000295711 2610
g__Gardnerella 1
UP000033124 0
g__Globicatella 1
UP000176615 2027


In [17]:
# Write the bacterial proteins to a txt
with open("./data/bacterial_proteins.txt", "w") as fp:
    for prot in proteins:
        fp.write(prot + "\n")

# Slice them in batches of 90000 and write them to different files
for i in range(0, len(proteins), 90000):
    prot_batch = list(proteins)[i:i+90000]
    with open("./data/bacterial_proteins_batch_" + str(i) + ".txt", "w") as fp:
        for prot in prot_batch:
            fp.write(prot + "\n")

*...*

**Wait for the Uniprot Mapping**

Under data/bac_mapping

**...**

In [37]:
df_list = list()
for mapping_file in os.listdir("./data/bac_mapping"):
    # Read all .tsv.gz files
    mapping_df = pd.read_csv("./data/bac_mapping/" + mapping_file, sep="\t", compression="gzip")
    # Append to df_list
    df_list.append(mapping_df)

bac_prots_df = pd.concat(df_list, axis=0)
bac_prots_df

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,R5N1R7,R5N1R7,unreviewed,R5N1R7_9FIRM,Elongation factor Ts (EF-Ts),tsf BN519_00791,Eubacterium sp. CAG:180,309,MAFTASDVKALREKTGVGMMDCKKALVESDGDMDKAIDFLREKGLA...
1,A0A173W6J8,A0A173W6J8,unreviewed,A0A173W6J8_9FIRM,Ribonuclease PH (RNase PH) (EC 2.7.7.56) (tRNA...,rph ERS852385_00019,Mitsuokella jalaludinii,248,MTRTDRRMADALRPYEIHRHFQKYPAGSVLIEMGNTKVICAASIED...
2,Q8R602,Q8R602,reviewed,EFG_FUSNN,Elongation factor G (EF-G),fusA FN1556,Fusobacterium nucleatum subsp. nucleatum (stra...,693,MARKISLDMTRNVGIMAHIDAGKTTTTERILFYTGVERKLKEVHEG...
3,C7HTX5,C7HTX5,unreviewed,C7HTX5_9FIRM,histidine kinase (EC 2.7.13.3),phoR HMPREF0078_0726,Anaerococcus vaginalis ATCC 51170,479,MEIEKKKKSLFKINFNSIIFSIMKIIMIFVTLVVVGFEIFSYRSIT...
4,R6V9Z1,R6V9Z1,unreviewed,R6V9Z1_9FIRM,Cell division protein SepF,sepF BN635_02018,Roseburia sp. CAG:380,193,MANPLGNLLQLLRLGDDEDYDDDYYEDDFEDDIDEEEERSLLAEEQ...
...,...,...,...,...,...,...,...,...,...
89995,R9HSW2,R9HSW2,unreviewed,R9HSW2_BACUN,Lon protease (EC 3.4.21.53) (ATP-dependent pro...,lon C801_02872,Bacteroides uniformis dnLKV2,826,MKKRNYLSDVEDRNGNAFSVIADFEGNEEQLMDIEVEEVLPILPLR...
89996,A0A6I3RXL6,A0A6I3RXL6,unreviewed,A0A6I3RXL6_9BURK,D-glycero-beta-D-manno-heptose 1-phosphate ade...,rfaE2 GMD42_01825,Parasutterella excrementihominis,164,MSQHLPPPPFEKKIVSFEKLEEFVRKLPHPVVLTNGVFDILHRGHV...
89997,Q8A4L3,Q8A4L3,unreviewed,Q8A4L3_BACTN,Uncharacterized protein,BT_2586,Bacteroides thetaiotaomicron (strain ATCC 2914...,167,MEEWIQYARDMAKAEKELEIELWVIISFYRTTEQGGKVLLFRYDLP...
89998,A0A173T8G7,A0A173T8G7,unreviewed,A0A173T8G7_9FIRM,Cell division topological specificity factor MinE,comes_20010 DW252_07705 DW656_00130 DWW65_0524...,Coprococcus comes,76,MHVFERGMHHSSVRIAKERLKSLLVSDRVQCQPDTYELLCGELYHT...


In [38]:
# Drop NaN for Sequence
bac_prots_df = bac_prots_df.dropna(subset=["Sequence"])
bac_prots_df = bac_prots_df.reset_index(drop=True)
bac_prots_df

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,R5N1R7,R5N1R7,unreviewed,R5N1R7_9FIRM,Elongation factor Ts (EF-Ts),tsf BN519_00791,Eubacterium sp. CAG:180,309,MAFTASDVKALREKTGVGMMDCKKALVESDGDMDKAIDFLREKGLA...
1,A0A173W6J8,A0A173W6J8,unreviewed,A0A173W6J8_9FIRM,Ribonuclease PH (RNase PH) (EC 2.7.7.56) (tRNA...,rph ERS852385_00019,Mitsuokella jalaludinii,248,MTRTDRRMADALRPYEIHRHFQKYPAGSVLIEMGNTKVICAASIED...
2,Q8R602,Q8R602,reviewed,EFG_FUSNN,Elongation factor G (EF-G),fusA FN1556,Fusobacterium nucleatum subsp. nucleatum (stra...,693,MARKISLDMTRNVGIMAHIDAGKTTTTERILFYTGVERKLKEVHEG...
3,C7HTX5,C7HTX5,unreviewed,C7HTX5_9FIRM,histidine kinase (EC 2.7.13.3),phoR HMPREF0078_0726,Anaerococcus vaginalis ATCC 51170,479,MEIEKKKKSLFKINFNSIIFSIMKIIMIFVTLVVVGFEIFSYRSIT...
4,R6V9Z1,R6V9Z1,unreviewed,R6V9Z1_9FIRM,Cell division protein SepF,sepF BN635_02018,Roseburia sp. CAG:380,193,MANPLGNLLQLLRLGDDEDYDDDYYEDDFEDDIDEEEERSLLAEEQ...
...,...,...,...,...,...,...,...,...,...
904549,R9HSW2,R9HSW2,unreviewed,R9HSW2_BACUN,Lon protease (EC 3.4.21.53) (ATP-dependent pro...,lon C801_02872,Bacteroides uniformis dnLKV2,826,MKKRNYLSDVEDRNGNAFSVIADFEGNEEQLMDIEVEEVLPILPLR...
904550,A0A6I3RXL6,A0A6I3RXL6,unreviewed,A0A6I3RXL6_9BURK,D-glycero-beta-D-manno-heptose 1-phosphate ade...,rfaE2 GMD42_01825,Parasutterella excrementihominis,164,MSQHLPPPPFEKKIVSFEKLEEFVRKLPHPVVLTNGVFDILHRGHV...
904551,Q8A4L3,Q8A4L3,unreviewed,Q8A4L3_BACTN,Uncharacterized protein,BT_2586,Bacteroides thetaiotaomicron (strain ATCC 2914...,167,MEEWIQYARDMAKAEKELEIELWVIISFYRTTEQGGKVLLFRYDLP...
904552,A0A173T8G7,A0A173T8G7,unreviewed,A0A173T8G7_9FIRM,Cell division topological specificity factor MinE,comes_20010 DW252_07705 DW656_00130 DWW65_0524...,Coprococcus comes,76,MHVFERGMHHSSVRIAKERLKSLLVSDRVQCQPDTYELLCGELYHT...


In [39]:
# Save 
bac_prots_df.to_csv("./data/bac_proteins.csv", index=False)

### Human protein (Gut and Brain Axis)

In [29]:
# Read gut and brain human data
gut = pd.read_csv(human_gut)
brain = pd.read_csv(human_brain)
# Concatenate gut and brain
gut_brain = pd.concat([gut, brain], axis=0)
# Drop duplicates
gut_brain = gut_brain.drop_duplicates()

# Get gene names
gene_names = gut_brain["Gene name"].unique()

# Write them to a txt
with open("./data/human_gene_names.txt", "w") as fp:
    for gene in gene_names:
        fp.write(gene + "\n")

*...*

**Wait for the Uniprot Mapping**

**...**

In [31]:
human_gut_brain = "./data/gut_c3/human_gut_brain.tsv.gz"

# Read gut and brain human data
gut_brain = pd.read_csv(human_gut_brain, sep="\t")
gut_brain

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,TSPAN6,O43657,reviewed,TSN6_HUMAN,Tetraspanin-6 (Tspan-6) (A15 homolog) (Putativ...,TSPAN6 TM4SF6 UNQ767/PRO1560,Homo sapiens (Human),245,MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...
1,TSPAN6,A0A087WYV6,unreviewed,A0A087WYV6_HUMAN,Tetraspanin-6,TSPAN6,Homo sapiens (Human),129,MLKLYAMFLTLVFLVELVAAIVGFVFRHEIKNSFKNNYEKALKQYN...
2,DPM1,O60762,reviewed,DPM1_HUMAN,Dolichol-phosphate mannosyltransferase subunit...,DPM1,Homo sapiens (Human),260,MASLEVSRSPRRSRRELEVRSPRQNKYSVLLPTYNERENLPLIVWL...
3,DPM1,A0A0S2Z4U1,unreviewed,A0A0S2Z4U1_HUMAN,Dolichol-phosphate mannosyltransferase subunit...,DPM1,Homo sapiens (Human),133,MASLEVSRSPRRSRRELEVRSPRQNKYSVLLPTYNERENLPLIVWL...
4,DPM1,A0A0S2Z4Y5,unreviewed,A0A0S2Z4Y5_HUMAN,Dolichol-phosphate mannosyltransferase subunit...,DPM1,Homo sapiens (Human),260,MASLEVSRSPRRSRRELEVRSPRQNKYSVLLPTYNERENLPLIVWL...
...,...,...,...,...,...,...,...,...,...
83381,GRIN2B,A0A0D9SFA0,unreviewed,A0A0D9SFA0_HUMAN,Glutamate ionotropic receptor NMDA type subuni...,GRIN2B,Homo sapiens (Human),35,MKPRAECCSPKFWLVLAVLAVSGSRARSQKSPPSI
83382,GRIN2B,A0A0D9SFK0,unreviewed,A0A0D9SFK0_HUMAN,Glutamate ionotropic receptor NMDA type subuni...,GRIN2B,Homo sapiens (Human),464,MKPRAECCSPKFWLVLAVLAVSGSRARSQKSPPSIGIAVILVGTSD...
83383,GRIN2B,A0A1B0GU78,unreviewed,A0A1B0GU78_HUMAN,Glutamate ionotropic receptor NMDA type subuni...,GRIN2B,Homo sapiens (Human),46,ANLAAFMIQEEYVDQVSGLSDKKMRTLRLRRSIFPKVTKLVTEPGF
83384,GRIN2B,A0A8D9PHB2,unreviewed,A0A8D9PHB2_HUMAN,Glutamate receptor,GRIN2B hCG_1811072,Homo sapiens (Human),1484,MKPRAECCSPKFWLVLAVLAVSGSRARSQKSPPSIGIAVILVGTSD...


In [32]:
# Keep only reviewed
gut_brain = gut_brain[gut_brain["Reviewed"] == "reviewed"].reset_index(drop=True)
gut_brain

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,TSPAN6,O43657,reviewed,TSN6_HUMAN,Tetraspanin-6 (Tspan-6) (A15 homolog) (Putativ...,TSPAN6 TM4SF6 UNQ767/PRO1560,Homo sapiens (Human),245,MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...
1,DPM1,O60762,reviewed,DPM1_HUMAN,Dolichol-phosphate mannosyltransferase subunit...,DPM1,Homo sapiens (Human),260,MASLEVSRSPRRSRRELEVRSPRQNKYSVLLPTYNERENLPLIVWL...
2,SCYL3,Q8IZE3,reviewed,PACE1_HUMAN,Protein-associating with the carboxyl-terminal...,SCYL3 PACE1,Homo sapiens (Human),742,MGSENSALKSYTLREPPFTLPSGLAVYPAVLQDGKFASVFVYKREN...
3,GCLC,P48506,reviewed,GSH1_HUMAN,Glutamate--cysteine ligase catalytic subunit (...,GCLC GLCL GLCLC,Homo sapiens (Human),637,MGLLSQGSPLSWEETKRHADHVRRHGILQFLHIYHAVKDRHKDVLK...
4,NFYA,P23511,reviewed,NFYA_HUMAN,Nuclear transcription factor Y subunit alpha (...,NFYA,Homo sapiens (Human),347,MEQYTANSNSSTEQIVVQAGQIQQQQQGGVTAVQLQTEAQVASASG...
...,...,...,...,...,...,...,...,...,...
10040,HS3ST5,Q96QI5,reviewed,HS3S6_HUMAN,Heparan sulfate glucosamine 3-O-sulfotransfera...,HS3ST6 HS3ST5,Homo sapiens (Human),342,MAGSGGLGGGAGGGQGAGAGQGAALRASRAPMLLVALVLGAYCLCA...
10041,YJEFN3,A6XGL0,reviewed,YJEN3_HUMAN,YjeF N-terminal domain-containing protein 3 (Y...,YJEFN3 AIBP2,Homo sapiens (Human),299,MSSAAGPDPSEAPEERHFLRALELQPPLADMGRAELSSNATTSLVQ...
10042,GPR162,Q16538,reviewed,GP162_HUMAN,Probable G-protein coupled receptor 162 (Gene-...,GPR162 GRCA,Homo sapiens (Human),588,MARGGAGAEEASLRSNALSWLACGLLALLANAWIILSISAKQQKHK...
10043,MIA,Q16674,reviewed,MIA_HUMAN,Melanoma-derived growth regulatory protein (Me...,MIA,Homo sapiens (Human),131,MARSLVCLGVIILLSAFSGPGVRGGPMPKLADRKLCADQECSHPIS...


In [33]:
# Remove duplicates
gut_brain = gut_brain.drop_duplicates(subset=["Entry"]).reset_index(drop=True)
gut_brain

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Sequence
0,TSPAN6,O43657,reviewed,TSN6_HUMAN,Tetraspanin-6 (Tspan-6) (A15 homolog) (Putativ...,TSPAN6 TM4SF6 UNQ767/PRO1560,Homo sapiens (Human),245,MASPSRRLQTKPVITCFKSVLLIYTFIFWITGVILLAVGIWGKVSL...
1,DPM1,O60762,reviewed,DPM1_HUMAN,Dolichol-phosphate mannosyltransferase subunit...,DPM1,Homo sapiens (Human),260,MASLEVSRSPRRSRRELEVRSPRQNKYSVLLPTYNERENLPLIVWL...
2,SCYL3,Q8IZE3,reviewed,PACE1_HUMAN,Protein-associating with the carboxyl-terminal...,SCYL3 PACE1,Homo sapiens (Human),742,MGSENSALKSYTLREPPFTLPSGLAVYPAVLQDGKFASVFVYKREN...
3,GCLC,P48506,reviewed,GSH1_HUMAN,Glutamate--cysteine ligase catalytic subunit (...,GCLC GLCL GLCLC,Homo sapiens (Human),637,MGLLSQGSPLSWEETKRHADHVRRHGILQFLHIYHAVKDRHKDVLK...
4,NFYA,P23511,reviewed,NFYA_HUMAN,Nuclear transcription factor Y subunit alpha (...,NFYA,Homo sapiens (Human),347,MEQYTANSNSSTEQIVVQAGQIQQQQQGGVTAVQLQTEAQVASASG...
...,...,...,...,...,...,...,...,...,...
9859,HS3ST5,Q96QI5,reviewed,HS3S6_HUMAN,Heparan sulfate glucosamine 3-O-sulfotransfera...,HS3ST6 HS3ST5,Homo sapiens (Human),342,MAGSGGLGGGAGGGQGAGAGQGAALRASRAPMLLVALVLGAYCLCA...
9860,YJEFN3,A6XGL0,reviewed,YJEN3_HUMAN,YjeF N-terminal domain-containing protein 3 (Y...,YJEFN3 AIBP2,Homo sapiens (Human),299,MSSAAGPDPSEAPEERHFLRALELQPPLADMGRAELSSNATTSLVQ...
9861,GPR162,Q16538,reviewed,GP162_HUMAN,Probable G-protein coupled receptor 162 (Gene-...,GPR162 GRCA,Homo sapiens (Human),588,MARGGAGAEEASLRSNALSWLACGLLALLANAWIILSISAKQQKHK...
9862,MIA,Q16674,reviewed,MIA_HUMAN,Melanoma-derived growth regulatory protein (Me...,MIA,Homo sapiens (Human),131,MARSLVCLGVIILLSAFSGPGVRGGPMPKLADRKLCADQECSHPIS...


In [34]:
# Save to csv
gut_brain.to_csv("./data/human_gut_brain_proteins.csv", index=False)

In [44]:
# Final stats

# Number of bacterial proteins
bac_proteins = bac_prots_df["Entry"].unique().tolist()
print("Number of bacterial proteins:", len(bac_proteins))

# Number of human proteins
human_proteins = gut_brain["Entry"].unique().tolist()
print("Number of human proteins:", len(human_proteins))

# Number of potential PPIs
print("Number of potential PPIs:", len(bac_proteins) * len(human_proteins))

Number of bacterial proteins: 904554
Number of human proteins: 9864
Number of potential PPIs: 8922520656
